In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q onnx onnxscript optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 10.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 4.9 MB/s eta 0:00:00:00:01


In [ ]:
import os
import json
import random
import torch
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Locate the official JSON files
# Note: Check the exact Kaggle input path, it is usually something like this:
base_dir = "/kaggle/input/datasets/mishalfatima09/facial-sketch-synthesis-dataset" 
train_json_path = os.path.join(base_dir, "anno_train.json")
test_json_path = os.path.join(base_dir, "anno_test.json")

# Load the official training annotations
with open(train_json_path, 'r') as f:
    train_anno = json.load(f)

all_train_pairs = []
labels = []

for item in train_anno:
    # Find the keys dynamically (usually 'image_name' and 'sketch_name')
    keys = list(item.keys())
    photo_key = next(k for k in keys if 'photo' in k.lower() or 'image' in k.lower())
    sketch_key = next(k for k in keys if 'sketch' in k.lower())
    
    # Infer the style (1, 2, or 3) directly from the folder name (photo1, photo2, photo3)
    photo_rel_path = item[photo_key]
    if 'photo1' in photo_rel_path or 'sketch1' in photo_rel_path:
        style_idx = 0  # Style 1 (0-indexed for PyTorch)
    elif 'photo2' in photo_rel_path or 'sketch2' in photo_rel_path:
        style_idx = 1  # Style 2
    else:
        style_idx = 2  # Style 3
        
    photo_path = os.path.join(base_dir, photo_rel_path)
    sketch_path = os.path.join(base_dir, item[sketch_key])
    
    all_train_pairs.append((photo_path, sketch_path, style_idx))
    labels.append(style_idx)

# 2. Stratified Validation Split (15%, Random Seed 42, Stratified by style)
train_data, val_data = train_test_split(
    all_train_pairs, 
    test_size=0.15, 
    random_state=42, 
    stratify=labels
)

# 3. Custom Dataset with IDENTICAL Spatial Augmentations
class FS2KDataset(Dataset):
    def __init__(self, data_list, is_train=True):
        self.data_list = data_list
        self.is_train = is_train

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        photo_path, sketch_path, style_label = self.data_list[idx]
        
        photo = Image.open(photo_path).convert('RGB')
        sketch = Image.open(sketch_path).convert('RGB')

        # Resize to 128x128
        photo = TF.resize(photo, (128, 128))
        sketch = TF.resize(sketch, (128, 128))

        # IDENTICAL augmentations (Both images must undergo the exact same transform!)
        if self.is_train:
            # Random Horizontal Flip
            if random.random() > 0.5:
                photo = TF.hflip(photo)
                sketch = TF.hflip(sketch)
                
            # Random slight rotation
            if random.random() > 0.5:
                angle = random.uniform(-10, 10)
                photo = TF.rotate(photo, angle)
                sketch = TF.rotate(sketch, angle)

        # Convert to Tensor and Normalize to [-1, 1] for GANs
        photo = TF.to_tensor(photo)
        sketch = TF.to_tensor(sketch)
        
        photo = TF.normalize(photo, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
        sketch = TF.normalize(sketch, [0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

        return photo, sketch, torch.tensor(style_label, dtype=torch.long)

# 4. Initialize DataLoaders
train_loader = DataLoader(FS2KDataset(train_data, is_train=True), batch_size=16, shuffle=True)
val_loader = DataLoader(FS2KDataset(val_data, is_train=False), batch_size=16, shuffle=False)

print(f"Loaded {len(train_data)} training pairs and {len(val_data)} validation pairs.")
p, s, label = next(iter(train_loader))
print(f"Batch Photo shape: {p.shape}, Sketch shape: {s.shape}, Labels: {label.shape}")